# Sparse Index Tracking — Experiment Analysis

Visualize results from the four experiments:
1. DFL vs Two-Stage
2. Sparsity-Accuracy Frontier
3. Factor vs Neural Model
4. Regime Analysis

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.insert(0, str(Path.cwd().parent))

from src.evaluation.plots import (
    plot_sparsity_frontier,
    plot_method_comparison,
    plot_training_curves,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 150

In [ ]:
# Load all result files
results_dir = Path("../results")
all_results = []

for f in results_dir.glob("*.json"):
    with open(f) as fh:
        data = json.load(fh)
        all_results.extend(data["results"])
        print(f"Loaded {f.name}: {len(data['results'])} entries")

df = pd.DataFrame(all_results)
df["method"] = df["model"] + " + " + df["loss"]
print(f"\nTotal results: {len(df)}")
df.head()

## Experiment 1: DFL vs Two-Stage

In [ ]:
# Filter to factor model results
factor_df = df[df["model"] == "factor"]

# Box plot: tracking error by loss type
fig = plot_method_comparison(
    factor_df.to_dict("records"),
    metric="tracking_error",
    save_path="../figures/exp1_te_comparison.png",
)
plt.show()

# Summary table
summary = factor_df.groupby("loss").agg({
    "tracking_error": ["mean", "std"],
    "avg_sparsity": "mean",
    "avg_turnover": "mean",
    "information_ratio": "mean",
}).round(4)
print(summary)

## Experiment 2: Sparsity-Accuracy Frontier

In [ ]:
fig = plot_sparsity_frontier(
    df.to_dict("records"),
    save_path="../figures/exp2_sparsity_frontier.png",
)
plt.show()

## Experiment 3: Factor vs Neural Model

In [ ]:
# Compare all four method combinations
fig = plot_method_comparison(
    df.to_dict("records"),
    metric="tracking_error",
    save_path="../figures/exp3_model_comparison.png",
)
plt.show()

# Per-K comparison
pivot = df.groupby(["K", "method"])["tracking_error"].mean().unstack()
print(pivot.round(4))

## Experiment 4: Regime Analysis

Split test periods by VIX regime (requires VIX data download or manual classification).

In [ ]:
# Aggregate metrics table
summary_all = df.groupby(["model", "loss"]).agg({
    "tracking_error": ["mean", "std"],
    "max_tracking_deviation": "mean",
    "avg_sparsity": "mean",
    "avg_turnover": "mean",
    "information_ratio": "mean",
}).round(4)

print("=== Full Results Summary ===")
print(summary_all)

# Save as LaTeX table
summary_all.to_latex("../figures/results_table.tex")
print("\nLaTeX table saved to figures/results_table.tex")